# C1.1 · Agentic offensive workflow

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *AI for Security*

Builds on **[B2.17 · Regression suites, cost curves and the economics of autonomy](https://spbreed.github.io/cyber-commons/lessons/B2.17.html)**.

| | |
|---|---|
| Open-source tooling | CAI, Metasploit, InterCode-CTF |
| Open-weight models | Kimi K2, GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Penetration testing has always been a loop: **recon → hypothesis → test →
escalate → report.** What has changed is who runs each turn.

**Manual (still the baseline).** A human runs `nmap`, reads the output, forms a
hypothesis, tries it. Slow, and the quality is entirely the tester's.

**Scripted.** The recon is automated — Nuclei templates, a Burp scan. The
hypothesis and the escalation are still human. This is where most teams are.

**Semi-autonomous.** An open-weight model reads the recon output and *proposes*
which findings are worth chasing and what to try next. The human approves each
action. The gain is triage speed on a large surface: 400 findings ranked in
minutes rather than a day.

**Autonomous.** The model proposes and the harness executes, within a
pre-approved scope and tool set, verifying its own results. This is real and it
works, and it is also where the engagement becomes a safety problem — because an
agent that has not understood the scope will happily test something outside it
at machine speed.

The professional obligations do not change with autonomy. They get harder,
because scope enforcement can no longer live in the tester's attention.

> **About the model in this notebook.** It runs offline against a deterministic
> replay so the lesson executes on a Kaggle kernel with no network. The replay
> is not a language model and is labelled as such. To run the same harness
> against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Demo — the four generations on the same recon output

Realistic scan output from an authorised engagement against hosts you own. The question at every generation is the same: what do I chase first?

In [ ]:
FINDINGS = [
 {"id": "F-01", "host": "api.target.example",   "port": 443, "svc": "https",
  "note": "TLS 1.0 enabled",                       "sev": "medium", "exploitable": False},
 {"id": "F-02", "host": "api.target.example",   "port": 443, "svc": "https",
  "note": "/v1/users returns data without auth",   "sev": "high",   "exploitable": True},
 {"id": "F-03", "host": "www.target.example",   "port": 80,  "svc": "http",
  "note": "server banner discloses version",       "sev": "low",    "exploitable": False},
 {"id": "F-04", "host": "api.target.example",   "port": 22,  "svc": "ssh",
  "note": "password auth permitted",               "sev": "medium", "exploitable": True},
 {"id": "F-05", "host": "legacy.target.example", "port": 8080, "svc": "http",
  "note": "directory listing enabled on /backup",  "sev": "medium", "exploitable": True},
 {"id": "F-06", "host": "cdn.partner.example",  "port": 443, "svc": "https",
  "note": "expired certificate",                   "sev": "low",    "exploitable": False},
]
SEV_RANK = {"low": 1, "medium": 2, "high": 3, "critical": 4}

print("=== generation 1: manual — a human reads all six and decides ===")
print(f"   {len(FINDINGS)} findings, no ordering, ~20 min of reading\n")

print("=== generation 2: scripted — sort by severity ===")
for f in sorted(FINDINGS, key=lambda x: -SEV_RANK[x["sev"]])[:3]:
    print(f"   {f['id']} {f['sev']:7s} {f['note']}")
print("   → severity is a label, not a prediction. F-04 and F-05 are both")
print("     'medium' and both actually exploitable; F-01 is not.")

In [ ]:
# === generation 3: semi-autonomous — the model proposes, the human approves ===
class ReplayModel:
    """DETERMINISTIC REPLAY — not a language model. See the note above."""
    ASSESSMENT = {
     "F-01": (0.10, "TLS 1.0 needs a downgrade position; no evidence of one here"),
     "F-02": (0.95, "unauthenticated data endpoint — directly exploitable, chase first"),
     "F-03": (0.05, "banner disclosure alone is not a finding worth engagement time"),
     "F-04": (0.60, "password auth permits spraying if no lockout; test lockout first"),
     "F-05": (0.75, "directory listing on /backup often exposes archives with secrets"),
     "F-06": (0.02, "expired cert on a partner CDN — out of scope, do not touch"),
    }
    def triage(self, f):
        conf, why = self.ASSESSMENT[f["id"]]
        return {"id": f["id"], "priority": conf, "reasoning": why}

model = ReplayModel()
ranked = sorted((model.triage(f) for f in FINDINGS), key=lambda r: -r["priority"])
print("=== generation 3: semi-autonomous triage ===")
for r in ranked:
    print(f"   {r['id']}  p={r['priority']:.2f}  {r['reasoning']}")

truth = {f["id"]: f["exploitable"] for f in FINDINGS}
top3 = [r["id"] for r in ranked[:3]]
print(f"\n   top 3 chosen: {top3}")
print(f"   of which actually exploitable: "
      f"{sum(truth[i] for i in top3)}/3")
sev_top3 = [f["id"] for f in sorted(FINDINGS, key=lambda x: -SEV_RANK[x['sev']])[:3]]
print(f"   severity-sorted top 3: {sev_top3} → "
      f"{sum(truth[i] for i in sev_top3)}/3 exploitable")

## 3 · Where it breaks — generation 4, and the scope problem

The model's top-ranked item is correct. Its reasoning on F-06 is also correct — *out of scope, do not touch*. Now make it autonomous and remove the human from the loop. What stops it acting on a finding it has correctly identified as out of scope?

Nothing in the model. Its judgement about scope is a *proposal*, on the decision plane, exactly like everything else it produces.

In [ ]:
ENGAGEMENT_SCOPE = {"api.target.example", "www.target.example",
                    "legacy.target.example"}

def autonomous_no_enforcement(findings, model):
    """The model's own scope judgement is the only control. This is the bug."""
    acted = []
    for f in findings:
        r = model.triage(f)
        if r["priority"] > 0.5:
            acted.append((f["id"], f["host"]))
    return acted

def autonomous_enforced(findings, model, scope):
    """Scope is enforced by the harness, not believed from the model."""
    acted, refused = [], []
    for f in findings:
        if f["host"] not in scope:
            refused.append((f["id"], f["host"], "host outside the engagement scope"))
            continue
        r = model.triage(f)
        if r["priority"] > 0.5:
            acted.append((f["id"], f["host"]))
    return acted, refused

acted = autonomous_no_enforcement(FINDINGS, model)
print("no enforcement — acted on:", acted)

acted2, refused = autonomous_enforced(FINDINGS, model, ENGAGEMENT_SCOPE)
print("\nenforced — acted on:", acted2)
for fid, host, why in refused:
    print(f"   REFUSED {fid} ({host}): {why}")

print("\nBoth runs happen to avoid F-06 here, because the model ranked it 0.02.")
print("The difference is that one of them would still avoid it if the model")
print("ranked it 0.99. That is the whole distinction between a judgement and a control.")

In [ ]:
# Verify: fuzz the model's scope judgement. The control must hold regardless.
import random
random.seed(4)

class AdversarialModel(ReplayModel):
    """A model that has been convinced F-06 is critical — by a prompt injection,
    a bad fine-tune, or simply by being wrong."""
    def triage(self, f):
        if f["id"] == "F-06":
            return {"id": "F-06", "priority": 0.99, "reasoning": "critical, chase now"}
        return super().triage(f)

bad = AdversarialModel()
acted_unsafe = autonomous_no_enforcement(FINDINGS, bad)
acted_safe, refused_safe = autonomous_enforced(FINDINGS, bad, ENGAGEMENT_SCOPE)

out_of_scope_unsafe = [i for i, h in acted_unsafe if h not in ENGAGEMENT_SCOPE]
out_of_scope_safe   = [i for i, h in acted_safe   if h not in ENGAGEMENT_SCOPE]
print(f"model convinced F-06 is critical:")
print(f"   unenforced → out-of-scope actions: {out_of_scope_unsafe}")
print(f"   enforced   → out-of-scope actions: {out_of_scope_safe}")
assert not out_of_scope_safe
print("\nScope enforcement in the harness is what makes autonomy professionally")
print("defensible. Without it, your engagement letter is protected by a prompt.")

## What you just proved

Severity sorting puts 2 of 3 exploitable findings in the top 3; model triage puts 3 of 3, and correctly reasons that the partner CDN is out of scope. With the model adversarially convinced that the out-of-scope host is critical, the unenforced harness acts on it and the enforced harness refuses.

## Your turn

Write your engagement scope as a data structure your harness reads, not as a paragraph in a PDF. Then ask what your current tooling would do if a target redirected to a host you were not authorised to touch.

---

**Next → [C1.2 · Sandboxing the offensive harness](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*